In [ ]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna.phy
except ImportError as e:
    import sys
    if 'google.colab' in sys.modules:
       # Install Sionna in Google Colab
       print("Installing Sionna and restarting the runtime. Please run the cell again.")
       os.system("pip install sionna")
       os.kill(os.getpid(), 5)
    else:
       raise e

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

# Set random seed for reproducability
sionna.phy.config.seed = 42

#matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from sionna.phy import Block
from sionna.phy.utils import ebnodb2no, compute_ser, compute_ber, PlotBER
from sionna.phy.channel import FlatFadingChannel, KroneckerModel
from sionna.phy.channel.utils import exp_corr_mat
from sionna.phy.mimo import lmmse_equalizer
from sionna.phy.mapping import SymbolDemapper, Mapper, Demapper, BinarySource, QAMSource
from sionna.phy.fec.ldpc.encoding import LDPC5GEncoder
from sionna.phy.fec.ldpc.decoding import LDPC5GDecoder

num_tx_ant = 8
num_rx_ant = 32
num_bits_per_symbol = 4
batch_size = 1024
qam_source = QAMSource(num_bits_per_symbol)
x = qam_source([batch_size, num_tx_ant])
print('x shape is : ',x.shape)

channel = FlatFadingChannel(num_tx_ant, num_rx_ant, add_awgn=True, return_channel=True)
no = 0.2 # Noise variance of the channel

# y and h are the channel output and channel realizations, respectively.
y, h = channel(x, no)
print('y shape is : ',y.shape)
print('h shape is : ',h.shape)

# dont run this one
h_hermitian = tf.linalg.adjoint(h)  # H^H (conjugate transpose)
print('H_hermitian shape is : ',h_hermitian.shape)
HtH = tf.matmul(h_hermitian, h)      # H^H H
print('H^H H IS :',HtH.shape)
eye = tf.eye(2 * self._num_tx_ant, batch_shape=[batch_size], dtype=self._real_dtype)
print('eye shape is : ',eye.shape)
A = HtH + no * eye
print('A=:',A.shape)
B=tf.matmul(h_hermitian, y_real[:, :, tf.newaxis])
print('B shape is : ',B.shape)

r=b
d=r
s=0
while r<1e-6 :
  for i in range(num_tx_ant):
    r_hermitian = tf.linalg.adjoint(r)
    print('r_hermitian shape is : ',r_hermitian.shape)
    rhr = tf.matmul(r_hermitian, r)
    print('rhr shape is : ',rhr.shape)
    rhrAd == tf.matmul(r_hermitian, A),tf.matmul(A, d)
    print('rhrAd shape is : ',rhrAd.shape)
    alpha=tf.math.divide(rhr,rhrAd)
    print('alpha shape is : ',alpha.shape)
    s_i=s+alpha*d
    print('s_i shape is : ',s_i.shape)
    r_i=r-alpha*A*d
    print('r_i shape is : ',r_i.shape)
    r_i_hermitian = tf.linalg.adjoint(r_i)
    print('r_i_hermitian shape is : ',r_i_hermitian.shape)
    r_ihr=tf.matmul(r_i_hermitian, r)
    print('r_ihr shape is : ',r_ihr.shape)
    beta=tf.math.divide(r_ihr,rhr)
    print('beta shape is : ',beta.shape)
    d_i=r_i+beta*d
    print('d_i shape is : ',d_i.shape)
    s=s_i
    r=r_i
    d=d_i
    return s

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sionna.phy.mapping import QAMSource  # Adjust import based on your library

class LCGNet:
    def __init__(self, num_tx_ant, num_rx_ant, complex_dtype=tf.complex64):
        self._num_tx_ant = num_tx_ant
        self._num_rx_ant = num_rx_ant
        self._complex_dtype = complex_dtype
        self._real_dtype = tf.float32

    def __call__(self, y, h, no, batch_size=None, max_iterations=100, tolerance=1e-6):
        if batch_size is None:
            batch_size = tf.shape(h)[0]
        xhat = self.compute(h, y, no, batch_size, max_iterations, tolerance)
        no = tf.convert_to_tensor(no, dtype=self._real_dtype)
        return xhat, no

    def compute(self, h, y, no, batch_size, max_iterations=100, tolerance=1e-6):
        h_hermitian = tf.linalg.adjoint(h)
        print('h_hermitian shape is :', h_hermitian.shape)

        HtH = tf.matmul(h_hermitian, h)
        print('HtH shape is :', HtH.shape)

        eye = tf.eye(self._num_tx_ant, batch_shape=[batch_size], dtype=self._complex_dtype)
        print('eye shape is :', eye.shape)

        A = HtH + tf.cast(no, self._complex_dtype) * eye
        print('A shape is :', A.shape)

        B = tf.matmul(h_hermitian, tf.expand_dims(y, axis=-1))
        print('B shape is :', B.shape)

        r = B
        print('r shape is :', r.shape)
        d = r
        print('d shape is :', d.shape)
        xhat = tf.zeros_like(B, dtype=self._complex_dtype)
        print('xhat shape is :', xhat.shape)

        for i in range(max_iterations):
            r_hermitian = tf.linalg.adjoint(r)
            print('r_hermitian shape is :', r_hermitian.shape)
            rhr = tf.matmul(r_hermitian, r)
            print('rhr shape is :', rhr.shape)

            Ad = tf.matmul(A, d)
            print('Ad shape is :', Ad.shape)
            rhrAd = tf.matmul(r_hermitian, Ad)
            print('rhrAd shape is :', rhrAd.shape)

            alpha = rhr / (rhrAd + tf.keras.backend.epsilon())
            print('alpha shape is :', alpha.shape)

            xhat = xhat + alpha * d
            print('xhat shape is :', xhat.shape)

            r_new = r - alpha * Ad
            print('r_new shape is :', r_new.shape)

            r_norm = tf.reduce_max(tf.reduce_sum(tf.square(tf.abs(r_new)), axis=[1, 2]))
            print('r_norm shape is :', r_norm.shape)
            print('r_norm value :', r_norm)
            if tf.cast(r_norm, self._real_dtype) < tolerance:
                print(f"Converged after {i+1} iterations")
                break

            r_new_hermitian = tf.linalg.adjoint(r_new)
            print('r_new_hermitian shape is :', r_new_hermitian.shape)
            r_new_hr = tf.matmul(r_new_hermitian, r_new)
            print('r_new_hr shape is :', r_new_hr.shape)
            beta = r_new_hr / (rhr + tf.keras.backend.epsilon())
            print('beta shape is :', beta.shape)

            d = r_new + beta * d
            print('d shape is :', d.shape)

            r = r_new
            print('r shape is :', r.shape)

        return tf.squeeze(xhat, axis=-1)

# Parameters
num_tx_ant = 4
num_rx_ant = 16
batch_size = 1024
no = 0.2
print('Noise variance (no):', no)

# Create inputs using QAMSource
num_bits_per_symbol = 2
qam_source = QAMSource(num_bits_per_symbol)
x = qam_source([batch_size, num_tx_ant])
print('x shape is :', x.shape)

# Debug: Print power of x
power_x = tf.reduce_mean(tf.square(tf.abs(x)))
print('Power of x:', power_x)

# Simulate channel and received signal
h = tf.complex(tf.random.normal([batch_size, num_rx_ant, num_tx_ant]),
               tf.random.normal([batch_size, num_rx_ant, num_tx_ant]))
y = tf.matmul(h, tf.expand_dims(x, axis=-1))
noise = tf.complex(tf.random.normal([batch_size, num_rx_ant, 1]),
                   tf.random.normal([batch_size, num_rx_ant, 1])) * tf.cast(tf.sqrt(no/2), tf.complex64)
y = y + noise
y = tf.squeeze(y, axis=-1)

# Instantiate and run detector
detector = LCGNet(num_tx_ant, num_rx_ant, complex_dtype=tf.complex64)
result = detector(y, h, no, max_iterations=500, tolerance=1e-8)
xhat, no_eff = result

# Debug: Inspect xhat after unpacking
print('Type of xhat:', type(xhat))
print('xhat shape after detector call:', xhat.shape)

# Print no_eff shape
print('no_eff shape:', no_eff.shape)

# xhat is already in complex form
xhat_complex = xhat

# Normalize xhat
power = tf.reduce_mean(tf.square(tf.abs(xhat_complex)))
print('Power of xhat_complex before normalization:', power)
xhat_complex = xhat_complex / tf.cast(tf.sqrt(power), tf.complex64)
power_after = tf.reduce_mean(tf.square(tf.abs(xhat_complex)))
print('Power of xhat_complex after normalization:', power_after)

# Compute noise variances
u = x - xhat_complex
noise_var_eff = np.var(u)
noise_var_est = np.mean(no_eff)
print('noise_var_eff:', noise_var_eff)
print('noise_var_est:', noise_var_est)

# Plot
plt.scatter(np.real(xhat_complex), np.imag(xhat_complex), color='orange', label='Estimated')
plt.scatter(np.real(x), np.imag(x), color='blue', alpha=0.5, label='True')
plt.xlabel('Real')
plt.ylabel('Imag')
plt.grid(True)
plt.legend()
plt.show()

print('Output shape:', xhat.shape)
print('Effective noise variance (no_eff):', no_eff)

print(no_eff.shape)

u=x-xhat_complex
noise_var_eff = np.var(u)
noise_var_est = np.mean(no_eff)
print(noise_var_eff)
print(noise_var_est)

symbol_demapper = SymbolDemapper("qam", num_bits_per_symbol, hard_out=True)

# Get symbol indices for the transmitted symbols
x_ind = symbol_demapper(x, no)

# Get symbol indices for the received soft-symbols
x_ind_hat = symbol_demapper(xhat, no)

compute_ser(x_ind, x_ind_hat)

# Create transmit and receive correlation matrices
r_tx = exp_corr_mat(0.4, num_tx_ant)
r_rx = exp_corr_mat(0.9, num_rx_ant)

# Add the spatial correlation model to the channel
channel.spatial_corr = KroneckerModel(r_tx, r_rx)



# Instantiate channel
channel = FlatFadingChannel(
    num_tx_ant=num_tx_ant,
    num_rx_ant=num_rx_ant,
    spatial_corr=None,  # Uncorrelated channel
    add_awgn=False,     # No AWGN for covariance computation
    return_channel=True
)

h = channel.generate(1000000)





# Compute empirical covariance matrices
r_tx_hat = tf.reduce_mean(tf.matmul(h, h, adjoint_a=True), 0) / num_rx_ant  # Shape: [8, 8]
r_rx_hat = tf.reduce_mean(tf.matmul(h, h, adjoint_b=True), 0) / num_tx_ant  # Shape: [32, 32]

# Theoretical covariance matrices (uncorrelated channel)
r_tx = tf.eye(num_tx_ant, dtype=tf.complex64)  # Shape: [8, 8]
r_rx = tf.eye(num_rx_ant, dtype=tf.complex64)  # Shape: [32, 32]


# Test that the empirical results match the theory
assert(np.allclose(r_tx, r_tx_hat, atol=1e-2))
assert(np.allclose(r_rx, r_rx_hat, atol=1e-2))

y, h = channel(x, no)
x_hat, no_eff = detector(y, h, no)
x_ind_hat = symbol_demapper(x_hat, no)
compute_ser(x_ind, x_ind_hat)

#fixed code of parameters:

n = 1024 # codeword length
k = 512  # number of information bits per codeword
coderate = k/n # coderate
batch_size = 32

binary_source = BinarySource()
encoder = LDPC5GEncoder(k, n)
decoder = LDPC5GDecoder(encoder, hard_out=True)
mapper = Mapper("qam", num_bits_per_symbol)
demapper = Demapper("app", "qam", num_bits_per_symbol)

b = binary_source([batch_size, num_tx_ant, k])
c = encoder(b)
x = mapper(c)
x_ind = symbol_demapper(x, no) # Get symbol indices for SER computation later on
shape = tf.shape(x)
x = tf.reshape(x, [-1, num_tx_ant])
print(x.shape)

y, h = channel(x, no)
x_hat, no_eff = detector(y, h, no)

x_ind_hat.shape

x_hat = tf.reshape(x_hat, shape)


llr = demapper(x_hat, no_eff)
b_hat = decoder(llr)

x_ind_hat = symbol_demapper(x_hat, no)
ber = compute_ber(b, b_hat).numpy()
print("Uncoded SER : {}".format(compute_ser(x_ind, x_ind_hat)))
print("Coded BER : {}".format(compute_ber(b, b_hat)))

ber_plot = PlotBER()

import tensorflow as tf
import numpy as np


class Model(Block):  # Assuming Block is a base class
    def __init__(self, spatial_corr=None):
        super().__init__()
        self.n = 1024
        self.k = 512
        self.coderate = self.k / self.n
        self.num_bits_per_symbol = 2  # 4-QAM
        self.num_tx_ant = 4
        self.num_rx_ant = 16
        self.batch_size = 1024
        self.no = 0.2  # Fixed noise variance (unused)

        # Sionna components
        self.binary_source = BinarySource()
        self.encoder = LDPC5GEncoder(self.k, self.n)
        self.mapper = Mapper("qam", self.num_bits_per_symbol)
        self.demapper = Demapper("app", "qam", self.num_bits_per_symbol)
        self.decoder = LDPC5GDecoder(self.encoder, hard_out=True)
        self.channel = FlatFadingChannel(
            self.num_tx_ant,
            self.num_rx_ant,
            spatial_corr=None,
            add_awgn=False,  # Manual AWGN to avoid Pack error
            return_channel=True
        )
        # LCGNet detector
        self.detector = LCGNet(self.num_tx_ant, self.num_rx_ant, complex_dtype=tf.complex64)

    def call(self, batch_size, ebno_db):
        # Generate data
        b = self.binary_source([batch_size, self.num_tx_ant, self.k])  # [batch_size, 4, 512]
        c = self.encoder(b)  # [batch_size, 4, 1024]
        x = self.mapper(c)  # [batch_size, 4, 512]
        print("x dtype:", x.dtype, "shape:", x.shape)
        x = tf.reshape(x, [-1, self.num_tx_ant])  # [batch_size * 512, 4]
        print("x reshaped dtype:", x.dtype, "shape:", x.shape)

        # Noise variance
        no = ebnodb2no(ebno_db, self.num_bits_per_symbol, self.coderate)
        no = tf.cast(no, tf.float32)  # Unscaled
        no_scaled = no * tf.sqrt(tf.cast(self.num_rx_ant, tf.float32))  # Scaled for LCGNet/demapper
        batch_size_total = batch_size * (self.n // self.num_bits_per_symbol)  # 4096 * 512 = 2097152
        print("no dtype:", no.dtype, "value:", no)
        print("no_scaled dtype:", no_scaled.dtype, "value:", no_scaled)

        # Channel (without AWGN)
        try:
            y, h = self.channel([x])  # No noise
            print("y dtype:", y.dtype, "shape:", y.shape)
            print("h dtype:", h.dtype, "shape:", h.shape)
        except Exception as e:
            print("Channel error:", e)
            raise

        # Remove extra batch dimension
        y = tf.squeeze(y, axis=0)  # (1, 2097152, 16) -> (2097152, 16)
        h = tf.expand_dims(tf.squeeze(h, axis=0), axis=0)  # (1, 16, 4) -> (1, 16, 4)
        h = tf.tile(h, [batch_size_total, 1, 1])  # (1, 16, 4) -> (2097152, 16, 4)
        print("y squeezed dtype:", y.dtype, "shape:", y.shape)
        print("h expanded dtype:", h.dtype, "shape:", h.shape)

        # Manual AWGN
        noise = tf.complex(
            tf.random.normal(tf.shape(y), stddev=tf.sqrt(no / 2), dtype=tf.float32),
            tf.random.normal(tf.shape(y), stddev=tf.sqrt(no / 2), dtype=tf.float32)
        )
        y = y + noise
        print("y with noise dtype:", y.dtype, "shape:", y.shape)

        # Ensure channel outputs are complex64
        y = tf.cast(y, tf.complex64)
        h = tf.cast(h, tf.complex64)

        # LCGNet detector
        xhat_output = self.detector(y, h, no_scaled)  # Expected: tuple (xhat, no)
        print("xhat_output type:", type(xhat_output), "contents:", [x.shape if hasattr(x, 'shape') else type(x) for x in xhat_output] if isinstance(xhat_output, tuple) else xhat_output.shape)
        # Extract xhat from tuple
        if isinstance(xhat_output, tuple):
            xhat = xhat_output[0]  # Assume first element is xhat
            print("xhat extracted dtype:", xhat.dtype, "shape:", xhat.shape)
        else:
            xhat = xhat_output
            print("xhat dtype:", xhat.dtype, "shape:", xhat.shape)

        # Normalize xhat
        power = tf.reduce_mean(tf.square(tf.abs(xhat)))
        print("Power of xhat before normalization:", power)
        xhat = xhat / tf.cast(tf.sqrt(power), tf.complex64)
        power_after = tf.reduce_mean(tf.square(tf.abs(xhat)))
        print("Power of xhat after normalization:", power_after)

        # Reshape xhat for demapper
        xhat = tf.reshape(xhat, [batch_size, self.num_tx_ant, self.n // self.num_bits_per_symbol])  # [4096, 4, 512]
        print("xhat reshaped dtype:", xhat.dtype, "shape:", xhat.shape)

        # Demapper
        try:
            print("no_scaled before demapper dtype:", no_scaled.dtype, "shape:", no_scaled.shape, "value:", no_scaled)
            llr = self.demapper(xhat, no_scaled)  # Correct syntax for sionna Demapper
            print("llr dtype:", llr.dtype, "shape:", llr.shape)
        except Exception as e:
            print("Demapper error:", e)
            raise

        # Reshape llr for decoder
        llr = tf.reshape(llr, [batch_size, self.num_tx_ant, self.n])  # [batch_size, 4, 1024]
        print("llr reshaped dtype:", llr.dtype, "shape:", llr.shape)

        # Decoder
        b_hat = self.decoder(llr)  # [batch_size, 4, 512]
        print("b_hat dtype:", b_hat.dtype, "shape:", b_hat.shape)
         
        return b, b_hat

# Example usage
if __name__ == "__main__":
    model1 = Model()
    try:
        ber_plot.simulate(
            model1,
            np.arange(-2.5, 0.25, 0.25),
            batch_size=4096,
            max_mc_iter=1000,
            num_target_block_errors=200,
            legend="Uncorrelated",
             
        )
    except Exception as e:
        print("Error:", e)

ber_plot = PlotBER()

r_tx = exp_corr_mat(0.4, num_tx_ant)
r_rx = exp_corr_mat(0.7, num_rx_ant)
model2 = Model(KroneckerModel(r_tx, r_rx))

ber_plot.simulate(model2,
        np.arange(0,2.6,0.25),
        batch_size=4096,
        max_mc_iter=1000,
        num_target_block_errors=200,
        legend="Kronecker model");

